<a href="https://colab.research.google.com/github/101academyforyou/Colab/blob/main/%E5%93%A1%E5%B7%A5%E8%88%87%E6%8E%92%E7%8F%AD%E7%AE%A1%E7%90%86%E7%B3%BB%E7%B5%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 員工與排班管理系統 (Employee and Schedule Management System)

In [8]:
import datetime # 導入 datetime 模組，用於處理日期和時間

# 定義員工類別 Employee
class Employee:
    # 初始化員工物件
    def __init__(self, employee_id, name, skills=None):
        self.employee_id = employee_id  # 員工的唯一ID
        self.name = name                # 員工姓名
        self.skills = skills if skills is not None else [] # 員工技能列表，如果未提供則為空列表

    # 定義物件的字串表示形式，方便列印員工資訊
    def __str__(self):
        return f"員工ID: {self.employee_id}, 姓名: {self.name}, 技能: {', '.join(self.skills)}"

# 定義班次類別 Shift
class Shift:
    # 初始化班次物件
    def __init__(self, shift_id, start_time, end_time, required_skills=None, assigned_employee=None):
        self.shift_id = shift_id        # 班次的唯一ID
        self.start_time = start_time    # 班次的開始時間 (datetime 物件)
        self.end_time = end_time        # 班次的結束時間 (datetime 物件)
        self.required_skills = required_skills if required_skills is not None else [] # 班次所需的技能列表
        self.assigned_employee = assigned_employee # 已分配的員工物件，初始為None表示未分配

    # 檢查班次是否可用 (即是否已分配員工)
    def is_available(self):
        return self.assigned_employee is None

    # 將員工分配給班次
    def assign_employee(self, employee):
        # 修正：原始邏輯有誤，不應將 required_skills 視為 employee_id 列表
        # 這裡應該檢查員工是否具備班次所需的所有技能
        # 為了簡化，目前只是直接分配，技能檢查在 ScheduleManager 中處理
        if self.is_available(): # 如果班次可用 (未分配)
            self.assigned_employee = employee # 分配員工
            return True
        return False # 如果班次不可用 (已分配)，則分配失敗

    # 定義物件的字串表示形式，方便列印班次資訊
    def __str__(self):
        # 取得已分配員工的姓名，如果沒有則顯示"未排定"
        employee_name = self.assigned_employee.name if self.assigned_employee else "未排定"
        return (
            f"班次ID: {self.shift_id}, "
            f"開始時間: {self.start_time.strftime('%Y-%m-%d %H:%M')}, " # 格式化開始時間
            f"結束時間: {self.end_time.strftime('%Y-%m-%d %H:%M')}, "   # 格式化結束時間
            f"需求技能: {', '.join(self.required_skills)}, "
            f"排定員工: {employee_name}"
        )


# 定義排班管理類別 ScheduleManager
class ScheduleManager:
    # 初始化排班管理系統
    def __init__(self):
        self.employees = {} # 儲存所有員工的字典，鍵為員工ID，值為Employee物件
        self.shifts = {}    # 儲存所有班次的字典，鍵為班次ID，值為Shift物件

    # 添加員工到系統中
    def add_employee(self, employee):
        self.employees[employee.employee_id] = employee

    # 添加班次到系統中
    def add_shift(self, shift):
        self.shifts[shift.shift_id] = shift

    # 嘗試將指定員工分配給指定班次
    def assign_shift(self, shift_id, employee_id):
        shift = self.shifts.get(shift_id)      # 根據ID獲取班次物件
        employee = self.employees.get(employee_id) # 根據ID獲取員工物件

        if not shift:
            print(f"錯誤: 未找到班次ID {shift_id}") # 如果班次不存在，列印錯誤訊息
            return False
        if not employee:
            print(f"錯誤: 未找到員工ID {employee_id}") # 如果員工不存在，列印錯誤訊息
            return False

        # 檢查員工是否具備班次所需的所有技能
        # all() 函數檢查迭代器中的所有元素是否為真
        if all(skill in employee.skills for skill in shift.required_skills):
            if shift.is_available(): # 如果班次可用 (未分配員工)
                shift.assign_employee(employee) # 執行分配
                print(f"成功: 員工 {employee.name} 已排定班次 {shift.shift_id}")
                return True
            else:
                print(f"錯誤: 班次 {shift.shift_id} 已排定員工") # 班次已被分配，列印錯誤
                return False
        else:
            # 員工不具備所需技能，列印錯誤訊息
            print(f"錯誤: 員工 {employee.name} 不具備班次 {shift.shift_id} 所需的技能")
            return False

    # 顯示當前所有班次的排班情況
    def display_schedule(self):
        print("\n--- 當前排班表 ---")
        if not self.shifts:
            print("目前沒有任何班次。") # 如果沒有班次，列印提示
            return
        for shift_id, shift in self.shifts.items():
            print(shift) # 列印每個班次的詳細資訊 (調用 Shift 的 __str__ 方法)

    # 顯示所有員工列表
    def display_employees(self):
        print("\n--- 員工列表 ---")
        if not self.employees:
            print("目前沒有任何員工。") # 如果沒有員工，列印提示
            return
        for employee_id, employee in self.employees.items():
            print(employee) # 列印每個員工的詳細資訊 (調用 Employee 的 __str__ 方法)


In [9]:
import json # 導入 json 模組，用於處理 JSON 資料
import csv  # 導入 csv 模組，用於處理 CSV 資料

# 將匯出功能添加到 ScheduleManager 類別中
class ScheduleManager(ScheduleManager):
    # 匯出員工資料至 JSON 檔案
    def export_employees_to_json(self, filename='employees.json'):
        employee_data = []
        for emp_id, employee in self.employees.items():
            employee_data.append({
                'employee_id': employee.employee_id,
                'name': employee.name,
                'skills': employee.skills
            })
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(employee_data, f, ensure_ascii=False, indent=4)
        print(f"員工資料已匯出至 {filename}")

    # 匯出班次資料至 JSON 檔案
    def export_shifts_to_json(self, filename='shifts.json'):
        shift_data = []
        for shift_id, shift in self.shifts.items():
            employee_name = shift.assigned_employee.name if shift.assigned_employee else "未排定"
            shift_data.append({
                'shift_id': shift.shift_id,
                'start_time': shift.start_time.isoformat(), # 將 datetime 物件轉換為 ISO 格式字串
                'end_time': shift.end_time.isoformat(),
                'required_skills': shift.required_skills,
                'assigned_employee': employee_name
            })
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(shift_data, f, ensure_ascii=False, indent=4)
        print(f"班次資料已匯出至 {filename}")

    # 匯出員工資料至 CSV 檔案
    def export_employees_to_csv(self, filename='employees.csv'):
        with open(filename, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['Employee ID', 'Name', 'Skills']) # 寫入標題行
            for emp_id, employee in self.employees.items():
                writer.writerow([employee.employee_id, employee.name, ', '.join(employee.skills)])
        print(f"員工資料已匯出至 {filename}")

    # 匯出班次資料至 CSV 檔案
    def export_shifts_to_csv(self, filename='shifts.csv'):
        with open(filename, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['Shift ID', 'Start Time', 'End Time', 'Required Skills', 'Assigned Employee'])
            for shift_id, shift in self.shifts.items():
                employee_name = shift.assigned_employee.name if shift.assigned_employee else "未排定"
                writer.writerow([
                    shift.shift_id,
                    shift.start_time.strftime('%Y-%m-%d %H:%M'),
                    shift.end_time.strftime('%Y-%m-%d %H:%M'),
                    ', '.join(shift.required_skills),
                    employee_name
                ])
        print(f"班次資料已匯出至 {filename}")


### 系統示範

現在，我們將示範如何使用這些類別來添加員工、創建班次並進行排班。

In [10]:
# 初始化排班管理系統物件
manager = ScheduleManager()

# 創建員工物件
emp1 = Employee("E001", "張三", ["收銀", "補貨"])
emp2 = Employee("E002", "李四", ["補貨", "清潔"])
emp3 = Employee("E003", "王五", ["收銀"])

# 將員工添加到排班管理系統中
manager.add_employee(emp1)
manager.add_employee(emp2)
manager.add_employee(emp3)

# 顯示當前系統中的員工列表
manager.display_employees()

# 創建班次物件
shift1 = Shift(
    "S001",
    datetime.datetime(2023, 10, 26, 9, 0), # 開始時間：2023年10月26日 09:00
    datetime.datetime(2023, 10, 26, 17, 0), # 結束時間：2023年10月26日 17:00
    ["收銀"] # 班次所需技能：收銀
)
shift2 = Shift(
    "S002",
    datetime.datetime(2023, 10, 26, 17, 0),
    datetime.datetime(2023, 10, 27, 1, 0),
    ["補貨", "清潔"] # 班次所需技能：補貨、清潔
)
shift3 = Shift(
    "S003",
    datetime.datetime(2023, 10, 27, 9, 0),
    datetime.datetime(2023, 10, 27, 17, 0),
    ["收銀"] # 班次所需技能：收銀
)

# 將班次添加到排班管理系統中
manager.add_shift(shift1)
manager.add_shift(shift2)
manager.add_shift(shift3)

# 顯示當前排班表（此時應顯示所有班次皆未排定員工）
manager.display_schedule()

# 進行排班操作
print("\n--- 進行排班 ---")
manager.assign_shift("S001", "E001") # 張三具備收銀技能，成功排定S001班次
manager.assign_shift("S002", "E002") # 李四具備補貨和清潔技能，成功排定S002班次
manager.assign_shift("S003", "E003") # 王五具備收銀技能，成功排定S003班次
manager.assign_shift("S001", "E002") # 嘗試將李四排定給S001班次，但S001已排定張三，應會失敗
manager.assign_shift("S003", "E002") # 嘗試將李四排定給S003班次，但李四不具備收銀技能，應會失敗

# 再次顯示排班表，確認排班結果
manager.display_schedule()



--- 員工列表 ---
員工ID: E001, 姓名: 張三, 技能: 收銀, 補貨
員工ID: E002, 姓名: 李四, 技能: 補貨, 清潔
員工ID: E003, 姓名: 王五, 技能: 收銀

--- 當前排班表 ---
班次ID: S001, 開始時間: 2023-10-26 09:00, 結束時間: 2023-10-26 17:00, 需求技能: 收銀, 排定員工: 未排定
班次ID: S002, 開始時間: 2023-10-26 17:00, 結束時間: 2023-10-27 01:00, 需求技能: 補貨, 清潔, 排定員工: 未排定
班次ID: S003, 開始時間: 2023-10-27 09:00, 結束時間: 2023-10-27 17:00, 需求技能: 收銀, 排定員工: 未排定

--- 進行排班 ---
成功: 員工 張三 已排定班次 S001
成功: 員工 李四 已排定班次 S002
成功: 員工 王五 已排定班次 S003
錯誤: 員工 李四 不具備班次 S001 所需的技能
錯誤: 員工 李四 不具備班次 S003 所需的技能

--- 當前排班表 ---
班次ID: S001, 開始時間: 2023-10-26 09:00, 結束時間: 2023-10-26 17:00, 需求技能: 收銀, 排定員工: 張三
班次ID: S002, 開始時間: 2023-10-26 17:00, 結束時間: 2023-10-27 01:00, 需求技能: 補貨, 清潔, 排定員工: 李四
班次ID: S003, 開始時間: 2023-10-27 09:00, 結束時間: 2023-10-27 17:00, 需求技能: 收銀, 排定員工: 王五


### 匯出資料示範

現在，我們將使用上述新增的匯出功能，將員工和班次資料匯出至 JSON 和 CSV 檔案。

In [11]:
# 匯出員工資料
manager.export_employees_to_json('employees_output.json')
manager.export_employees_to_csv('employees_output.csv')

# 匯出班次資料
manager.export_shifts_to_json('shifts_output.json')
manager.export_shifts_to_csv('shifts_output.csv')

print("\n--- 檢查匯出的檔案內容 ---")

# 讀取並顯示 JSON 檔案內容
with open('employees_output.json', 'r', encoding='utf-8') as f:
    print("employees_output.json 內容:")
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

with open('shifts_output.json', 'r', encoding='utf-8') as f:
    print("\nshifts_output.json 內容:")
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# 讀取並顯示 CSV 檔案內容 (只顯示前幾行)
print("\nemployees_output.csv 內容 (前5行):")
with open('employees_output.csv', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 5: break
        print(line.strip())

print("\nshifts_output.csv 內容 (前5行):")
with open('shifts_output.csv', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 5: break
        print(line.strip())


員工資料已匯出至 employees_output.json
員工資料已匯出至 employees_output.csv
班次資料已匯出至 shifts_output.json
班次資料已匯出至 shifts_output.csv

--- 檢查匯出的檔案內容 ---
employees_output.json 內容:
[
  {
    "employee_id": "E001",
    "name": "張三",
    "skills": [
      "收銀",
      "補貨"
    ]
  },
  {
    "employee_id": "E002",
    "name": "李四",
    "skills": [
      "補貨",
      "清潔"
    ]
  },
  {
    "employee_id": "E003",
    "name": "王五",
    "skills": [
      "收銀"
    ]
  }
]

shifts_output.json 內容:
[
  {
    "shift_id": "S001",
    "start_time": "2023-10-26T09:00:00",
    "end_time": "2023-10-26T17:00:00",
    "required_skills": [
      "收銀"
    ],
    "assigned_employee": "張三"
  },
  {
    "shift_id": "S002",
    "start_time": "2023-10-26T17:00:00",
    "end_time": "2023-10-27T01:00:00",
    "required_skills": [
      "補貨",
      "清潔"
    ],
    "assigned_employee": "李四"
  },
  {
    "shift_id": "S003",
    "start_time": "2023-10-27T09:00:00",
    "end_time": "2023-10-27T17:00:00",
    "required_skills": [
      "收

### 匯出功能完成

您現在已經可以將員工和班次資料匯出到 JSON 或 CSV 檔案了。這些檔案會儲存在 Colab 環境的根目錄下。

如果您想進行其他操作，例如匯入資料、更複雜的查詢或排班演算法，請告訴我您的需求！

## 互動式選單 (Interactive Menu)

In [12]:
def show_menu():
    print("\n--- 員工與排班管理系統選單 ---")
    print("1. 添加員工")
    print("2. 添加班次")
    print("3. 排定班次")
    print("4. 顯示所有員工")
    print("5. 顯示當前排班表")
    print("6. 匯出員工資料 (JSON)")
    print("7. 匯出班次資料 (JSON)")
    print("8. 匯出員工資料 (CSV)")
    print("9. 匯出班次資料 (CSV)")
    print("0. 結束程式")
    print("-------------------------")

def add_employee_interactive(manager):
    employee_id = input("請輸入員工ID: ")
    name = input("請輸入員工姓名: ")
    skills_str = input("請輸入員工技能 (用逗號分隔，例如: 收銀,補貨): ")
    skills = [s.strip() for s in skills_str.split(',') if s.strip()]
    new_employee = Employee(employee_id, name, skills)
    manager.add_employee(new_employee)
    print(f"員工 {name} ({employee_id}) 已添加。")

def add_shift_interactive(manager):
    shift_id = input("請輸入班次ID: ")
    start_time_str = input("請輸入班次開始時間 (YYYY-MM-DD HH:MM): ")
    end_time_str = input("請輸入班次結束時間 (YYYY-MM-DD HH:MM): ")
    required_skills_str = input("請輸入班次所需技能 (用逗號分隔): ")
    required_skills = [s.strip() for s in required_skills_str.split(',') if s.strip()]
    try:
        start_time = datetime.datetime.strptime(start_time_str, '%Y-%m-%d %H:%M')
        end_time = datetime.datetime.strptime(end_time_str, '%Y-%m-%d %H:%M')
        new_shift = Shift(shift_id, start_time, end_time, required_skills)
        manager.add_shift(new_shift)
        print(f"班次 {shift_id} 已添加。")
    except ValueError:
        print("時間格式錯誤，請使用 YYYY-MM-DD HH:MM 格式。")

def assign_shift_interactive(manager):
    shift_id = input("請輸入要排定的班次ID: ")
    employee_id = input("請輸入要排定的員工ID: ")
    manager.assign_shift(shift_id, employee_id)

# 重新初始化 manager 以便從乾淨的狀態開始，或繼續使用之前填充的 manager
# manager = ScheduleManager()

# 為了方便示範，我們先將之前創建的員工和班次重新加載到 manager 中
# (實際應用中，您會從文件或資料庫載入)
manager_for_menu = ScheduleManager() # 使用一個新的 manager 來避免重複操作
manager_for_menu.add_employee(Employee("E001", "張三", ["收銀", "補貨"]))
manager_for_menu.add_employee(Employee("E002", "李四", ["補貨", "清潔"]))
manager_for_menu.add_employee(Employee("E003", "王五", ["收銀"]))

manager_for_menu.add_shift(Shift("S001", datetime.datetime(2023, 10, 26, 9, 0), datetime.datetime(2023, 10, 26, 17, 0), ["收銀"]))
manager_for_menu.add_shift(Shift("S002", datetime.datetime(2023, 10, 26, 17, 0), datetime.datetime(2023, 10, 27, 1, 0), ["補貨", "清潔"]))
manager_for_menu.add_shift(Shift("S003", datetime.datetime(2023, 10, 27, 9, 0), datetime.datetime(2023, 10, 27, 17, 0), ["收銀"]))

# 排定一些班次以供示範
manager_for_menu.assign_shift("S001", "E001")
manager_for_menu.assign_shift("S002", "E002")
manager_for_menu.assign_shift("S003", "E003")

while True:
    show_menu()
    choice = input("請選擇一個選項: ")

    if choice == '1':
        add_employee_interactive(manager_for_menu)
    elif choice == '2':
        add_shift_interactive(manager_for_menu)
    elif choice == '3':
        assign_shift_interactive(manager_for_menu)
    elif choice == '4':
        manager_for_menu.display_employees()
    elif choice == '5':
        manager_for_menu.display_schedule()
    elif choice == '6':
        manager_for_menu.export_employees_to_json()
    elif choice == '7':
        manager_for_menu.export_shifts_to_json()
    elif choice == '8':
        manager_for_menu.export_employees_to_csv()
    elif choice == '9':
        manager_for_menu.export_shifts_to_csv()
    elif choice == '0':
        print("程式結束。感謝使用！")
        break
    else:
        print("無效的選項，請重新輸入。")


成功: 員工 張三 已排定班次 S001
成功: 員工 李四 已排定班次 S002
成功: 員工 王五 已排定班次 S003

--- 員工與排班管理系統選單 ---
1. 添加員工
2. 添加班次
3. 排定班次
4. 顯示所有員工
5. 顯示當前排班表
6. 匯出員工資料 (JSON)
7. 匯出班次資料 (JSON)
8. 匯出員工資料 (CSV)
9. 匯出班次資料 (CSV)
0. 結束程式
-------------------------
請選擇一個選項: 5

--- 當前排班表 ---
班次ID: S001, 開始時間: 2023-10-26 09:00, 結束時間: 2023-10-26 17:00, 需求技能: 收銀, 排定員工: 張三
班次ID: S002, 開始時間: 2023-10-26 17:00, 結束時間: 2023-10-27 01:00, 需求技能: 補貨, 清潔, 排定員工: 李四
班次ID: S003, 開始時間: 2023-10-27 09:00, 結束時間: 2023-10-27 17:00, 需求技能: 收銀, 排定員工: 王五

--- 員工與排班管理系統選單 ---
1. 添加員工
2. 添加班次
3. 排定班次
4. 顯示所有員工
5. 顯示當前排班表
6. 匯出員工資料 (JSON)
7. 匯出班次資料 (JSON)
8. 匯出員工資料 (CSV)
9. 匯出班次資料 (CSV)
0. 結束程式
-------------------------
請選擇一個選項: 4

--- 員工列表 ---
員工ID: E001, 姓名: 張三, 技能: 收銀, 補貨
員工ID: E002, 姓名: 李四, 技能: 補貨, 清潔
員工ID: E003, 姓名: 王五, 技能: 收銀

--- 員工與排班管理系統選單 ---
1. 添加員工
2. 添加班次
3. 排定班次
4. 顯示所有員工
5. 顯示當前排班表
6. 匯出員工資料 (JSON)
7. 匯出班次資料 (JSON)
8. 匯出員工資料 (CSV)
9. 匯出班次資料 (CSV)
0. 結束程式
-------------------------
請選擇一個選項: 0
程式結束。感謝使用！


### 下一步

這個基礎系統可以進一步擴展，例如：

*   **更複雜的排班邏輯：** 考慮員工可用性、偏好、工時限制等。
*   **資料儲存：** 將員工和班次資料保存到文件（CSV, JSON）或資料庫中，以便持久化。
*   **使用者介面：** 開發圖形化使用者介面 (GUI) 或更互動的命令列介面。
*   **衝突檢測：** 檢測員工工時衝突或技能不匹配等問題。

請告訴我您希望進一步開發哪個方面，或者有其他任何需求！